Все задания изначально выполнены в формате отдельного модуля с классами и методами, поэтому сюда я их перенесу в порядке некольцевого вызова. Разобью каждый блок Jupyter так, будто это отдельный файл

# Кодовая база

`homework1/core`

In [11]:
class NegativeDepositError(RuntimeError):
    """Введена отрицательная сумма депозита, что должно трактоваться как Critical Verbose ошибка"""


In [12]:
from collections.abc import Iterable, Sequence
from dataclasses import asdict, fields, is_dataclass
from datetime import date, datetime
from textwrap import shorten
from typing import Any


class TableFormatter:
    def __init__(
        self, class_: type, *, min_col_size: int = 6, max_col_size: int = 28
    ) -> None:
        """Форматтер для значений конкретного дата-класса в виде таблицы

        :param class_: Тип дата-класса
        :param min_col_size: Минимальный размер колонок. Должен быть больше или равен 6
        :param max_col_size: Максимальный размер колонок
        """
        assert is_dataclass(class_), "Передаваемое значение должно быть дата-классом"
        assert min_col_size >= 6, "Минимальная ширина колонки должна быть не меньше 6"

        self._class = class_

        self._col_size_space_indent = 2

        self._min_col_size = min_col_size
        self._max_col_size = max_col_size - self._col_size_space_indent

        self._row_vr_delim = "+"
        self._row_hr_delim = "-"
        self._col_vr_delim = "|"

        self._column_names: list[str] = [f.name for f in fields(class_)]
        self._column_types: list[str | Any] = [f.type for f in fields(class_)]
        self._column_sizes: list[int] = [0] * len(self._column_names)

    def _calc_columns(self, items: Iterable[dict]) -> None:
        """Не самая оптимальная функция подсчёта размера столбцов"""
        for item in items:
            for k, v in item.items():
                idx = self._column_names.index(k)

                size = max(len(str(v)), len(k)) if v else len(k)
                size += self._col_size_space_indent

                self._column_sizes[idx] = min(
                    max(size, self._column_sizes[idx], self._min_col_size),
                    self._max_col_size,
                )

    def _make_hr_line(self) -> str:
        """Метод генерации горизонтальной линии"""
        result = self._col_vr_delim

        for col_size in self._column_sizes:
            result += self._row_hr_delim * col_size + self._row_vr_delim

        return result

    def _make_header(self) -> str:
        """Метод генерации заголовка таблицы"""
        result = self._make_hr_line() + "\n"
        result += self._col_vr_delim

        for col_name, col_size in zip(
            self._column_names, self._column_sizes, strict=True
        ):
            col_name = shorten(col_name, col_size)
            result += f"{col_name:^{col_size}}" + self._col_vr_delim

        return result + "\n" + self._make_hr_line()

    def _make_row(self, item: dict) -> str:
        """Метод генерации строки таблицы"""
        result = self._col_vr_delim

        for k, v in item.items():
            idx = self._column_names.index(k)

            col_size = self._column_sizes[idx]
            col_type = self._column_types[idx]

            indent_ = "<"

            if isinstance(v, bool):
                indent_ = "^"
                v = str(v)

            if isinstance(v, int | float):
                indent_ = ">"

            # Я знаю, что здесь мог бы вызвать свой же кодек,
            # но по-хорошему для принтинга должен быть свой коде
            if isinstance(v, datetime):
                indent_ = ">"
                v = v.strftime("%Y-%m-%d %H:%M:%S")

            if isinstance(v, date):
                indent_ = ">"
                v = v.isoformat()

            if isinstance(v, str):
                v = shorten(v, col_size)

            if (isinstance(v, str) and not v) or v is None:
                v = ""

            result += f"{v:{indent_}{col_size}}" + self._col_vr_delim

        return result + "\n" + self._make_hr_line()

    def format(self, items: Sequence[Any]) -> str:
        """Форматирование списка значений в виде таблицы

        :param items: Список значений (все значения должны быть валидными дата-классами)
        :return: Строка-таблица
        """
        if not items:
            return ""

        # Да, это не самый точной способ определения, но fail-fast
        if not is_dataclass(items[0]):
            err = "Переданный список должен содержать дата-классы в качестве значений"
            raise ValueError(err)

        items = [asdict(i) for i in items]

        self._calc_columns(items)

        result = self._make_header() + "\n"

        for item in items:
            result += self._make_row(item) + "\n"

        return result


`homework1/operations`

In [13]:
from dataclasses import dataclass, field
from datetime import UTC, datetime
from functools import partial, wraps
from typing import Literal

from homework1.core.exc import NegativeDepositError

# Всё по PEP
utcnowfn = partial(datetime.now, UTC)

# Доступные типы операций на уровне аннотаций
OperationStatus = Literal["success", "fail"]


@dataclass
class Operation:
    """Класс для сохраняемых в истории операций"""

    # Название операции
    name: str = field(metadata={"serialization_alias": "operation"})

    # Статус операции
    status: OperationStatus = field()

    # Значение, передаваемое в операции
    value: float = field(metadata={"serialization_alias": "amount"})

    # Баланс аккаунта до выполнения операции
    balance_before: float | None = field(
        default=None, metadata={"serialize_field": False}
    )

    # Баланс аккаунта после выполнения операции.
    # None - поскольку операция может завершиться с ошибкой
    balance_after: float | None = None

    # Дата создания операции
    created_at: datetime = field(
        default_factory=utcnowfn, metadata={"serialization_alias": "date"}
    )


def mark_operation(op_name: str, *, key: str = "_operations_history"):
    """Декоратор с параметрами для записи операции в историю

    * "Глушить" в контексте этой функции - означает не передавать ошибку наверх, а решать её в рамках текущей функции

    :param op_name: Название операции
    :param key: Ключ, в котором хранится история в рамках текущего объекта
    """

    def decorator(fn):
        @wraps(fn)
        def wrapped(self, *args, **kwargs):
            # Управленческая конструкция,
            # чтобы вернуть ошибку, а не зашлушить её
            propagate_exc = kwargs.pop("__propagate_exc", False)

            value = None
            if args:
                value = args[0]
            elif kwargs:
                value = next(iter(kwargs.values()))

            op = Operation(
                name=op_name,
                status="fail",
                balance_before=self.get_balance(),
                value=value,  # Условно считаем, что значение будет всегда передано
            )

            try:
                result = fn(self, *args, **kwargs)

                op.status = "success"

                return result
            except NegativeDepositError:
                # Тк нам тут ошибка на логическом уровне;
                # мы просто хотим прокинуть её наверх
                raise
            except Exception:
                op.status = "fail"

                # Если мы не должны глушить ошибку, то возвращаем её
                if propagate_exc:
                    raise
            finally:
                op.balance_after = self.get_balance()

                # Поскольку это блок finally,
                # то нам необходимо дополнительно проверить, что мы не хотим глушить ошибку.
                # Если мы её всё же глушим, то добавляем запись об ошибке в историю
                if not propagate_exc:
                    getattr(self, key).append(op)

        return wrapped

    return decorator


In [14]:
from homework1.core.formatters import TableFormatter
from homework1.operations.operations import Operation


class OperationsTableFormatter(TableFormatter):
    def __init__(self, *, max_col_size: int = 28) -> None:
        """Форматтер для дата-класса Operation в виде таблицы"""
        super().__init__(class_=Operation, max_col_size=max_col_size)


In [15]:
import csv
import json
from abc import ABC, abstractmethod
from collections import defaultdict
from collections.abc import Generator, Sequence
from dataclasses import asdict, fields
from datetime import date, datetime
from os import PathLike
from pathlib import Path
from pprint import pformat
from typing import Any, NamedTuple

from homework1.operations.operations import Operation


class OperationCodec(ABC):
    """Кодек для сериализации операций"""

    @classmethod
    def encode(cls, value):
        """Простая функция дополнительного кодирования значений"""
        if isinstance(value, datetime):
            return value.strftime("%Y-%m-%d %H:%M:%S")

        if isinstance(value, date):
            return value.isoformat()

        return value

    @classmethod
    def decode(cls, value, type: str | Any = None):
        """Простая функция дополнительного декодирования значений"""
        # Поскольку традиционно для сериализации-десериализации даты и времени
        # используют три подхода:
        # - перебор
        # - типы внутри
        # - типы в коде
        # То я решил воспользоваться последним вариантом, в связи с его простотой и тем,
        # что мои типы уже содержатся в аннотациях полей датакласса
        if type in (datetime, date):
            try:
                result = datetime.strptime(value, "%Y-%m-%d %H:%M:%S")
                return result
            except:
                ...

            try:
                result = date.fromisoformat(value)
                return result
            except:
                raise

        # Обработка краевого кейса, тк уже два ночи и я не могу думать
        if value is None and type == float | None:
            return None

        if type in (float, float | None):
            return float(value)

        return value

    @classmethod
    def serialize(cls, value: Operation) -> dict:
        """Сериализация операции"""
        fields_ = {field_.name: field_ for field_ in fields(Operation)}

        kwargs = {}
        for k, v in asdict(value).items():
            nk = k

            if fields_[k].metadata:
                nk = fields_[k].metadata.get("serialization_alias", k)

                if not fields_[k].metadata.get("serialize_field", True):
                    nk = None

            if not nk:
                continue

            kwargs[nk] = cls.encode(v)

        return kwargs

    @classmethod
    def deserialize(cls, value: dict) -> Operation:
        """Десериализация операции"""
        kwargs = {}
        for k, v in value.items():
            nk = None
            for field_ in fields(Operation):
                if field_.name == k or (
                    field_.metadata
                    and field_.metadata.get("serialization_alias", None) == k
                ):
                    nk = field_.name
                    break

            if not nk:
                continue

            try:
                kwargs[nk] = cls.decode(v, type=field_.type)
            except ValueError as e:
                err = f"Произошла ошибка во время декодирования значения для поля {nk}={v}"
                raise ValueError(err) from e

        return Operation(**kwargs)


class AccountOperations(NamedTuple):
    """Дополнительный тип для более удобного представления операций,
    относящихся к конкретному аккаунту
    """

    account_number: str
    account_type: str
    operations: Sequence[Operation]


class OperationsHistoryFSStorageHandler(ABC):
    def __init__(self, codec: type[OperationCodec] = OperationCodec) -> None:
        """Базовый класс для сохранения или загрузки историй операций из файлов на файловой системе

        :param codec: Кодек для сериализации-десериализации значений
        """
        self.__codec = codec

    @property
    def codec(self) -> type[OperationCodec]:
        """Кодек для сериализации-десериализации значений"""
        return self.__codec

    @abstractmethod
    def _load(self, filepath: str | PathLike | Path) -> list[dict]:
        """Базовый метод загрузки данных по пути. Должен быть переопределён"""
        raise NotImplementedError

    @abstractmethod
    def _dump(self, filepath: str | PathLike | Path, history: list[dict]) -> None:
        """Базовый метод сохранения данных по пути. Должен быть переопределён"""
        raise NotImplementedError

    def _clean_history(
        self,
        history: Sequence[dict],
        *,
        account_type: str,
        allowed_operations: set[str],
    ) -> Generator[dict]:
        raise NotImplementedError(
            "Этот метод больше не нужен из-за избыточности, поскольку согласование теперь идёт на более высоком уровне"
        )

    def load(
        self,
        filepath: str | PathLike | Path,
    ) -> list[AccountOperations]:
        """Метод загрузки операций по аккаунтам из файла

        :param filepath: Путь к файлу с операциями
        :return: Список из операций, разделённых по конкретным аккаунтам
        """
        result = defaultdict(list)

        for raw_op in self._load(filepath=filepath):
            key = (raw_op.get("account_number"), raw_op.get("account_type"))

            try:
                result[key].append(self.codec.deserialize(raw_op))
            except Exception:
                print(f"Операцию не удалось распознать:\n{pformat(raw_op)}")

        return [AccountOperations(*k, v) for k, v in result.items()]

    def dump(
        self,
        filepath: str | PathLike | Path,
        history: list[AccountOperations],
    ) -> None:
        """Метод сохранения операций по аккаунтам в файл

        :param filepath: Путь к итоговому файлу
        :param history: Список из операций, относящихся к конкретному аккаунту
        """
        data = []

        for account_number, account_type, operations in history:
            for op in operations:
                data.append(
                    {
                        "account_number": account_number,
                        "account_type": account_type,
                        **self.codec.serialize(op),
                    }
                )

        self._dump(filepath, data)


class OperationsHistoryJSONFSStorageHandler(OperationsHistoryFSStorageHandler):
    """Сохранение и загрузка истории операций из JSON-файла"""

    def _load(self, filepath: str | PathLike | Path) -> list[dict]:
        with Path(filepath).open(mode="rb") as fp:
            return json.load(fp)

    def _dump(self, filepath: str | PathLike | Path, history: list[dict]) -> None:
        with Path(filepath).open(mode="w+") as fp:
            json.dump(history, fp)


class OperationsHistoryCSVFSStorageHandler(OperationsHistoryFSStorageHandler):
    """Сохранение и загрузка истории операций из CSV-файла"""

    def _load(self, filepath: str | PathLike | Path) -> list[dict]:
        with Path(filepath).open(mode="r", newline="") as fp:
            reader = csv.DictReader(fp)
            return list(reader)

    def _dump(self, filepath: str | PathLike | Path, history: list[dict]) -> None:
        assert history, "История операций должна быть передана для записи в файл"

        with Path(filepath).open(mode="w+") as fp:
            writer = csv.DictWriter(fp, fieldnames=history[0].keys())

            writer.writeheader()
            writer.writerows(history)


`homework1/accounts`

In [16]:
class AccountLimitReached(ValueError):
    """Ошибка, обозначающая, что превышен лимит доступных аккаунтов"""


class AccountNumbersManager:
    def __init__(self, *, limit: int = 1_000_000) -> None:
        """Менеджер номеров аккаунтов.

        Необходим для отслеживания созданных аккаунтов и корректной выдачи новых номеров.
        Важно понимать, что менеджер отслеживает дубликаты, чтобы не было двух одинаковых аккаунтов

        :param limit: Максимальный размер хранилища для номеров
        """
        self.__occupied_ids = set()
        self.__counter = 0

        self.__limit = limit

        self.__pattern = "ACC-{:04}"

    def __extract_number(self, value: str) -> int:
        """Получение номера аккаунта из строкового значения"""
        return int(value.split("-", maxsplit=1)[1])

    def _inc(self) -> int:
        """Получение следующего доступного номера.

        Метод является недетерминированным,
        т.е. его повторный вызов даст другое значение
        и будет иметь дальнейшие эффекты на работу всей системы

        :raises AccountLimitReached: Превышен лимит доступных аккаунтов
        """
        while self.__counter < self.__limit and self.__counter in self.__occupied_ids:
            self.__counter += 1

        if self.__counter >= self.__limit:
            err = "Превышен лимит доступных аккаунтов"
            raise AccountLimitReached(err)

        return self.__counter

    @property
    def pattern(self) -> str:
        """Строчный шаблон номера аккаунта"""
        return self.__pattern

    def addgen(self, number: int | str | None = None) -> str:
        """Добавление или генерация номера аккаунта.

        Если параметр передан, то будет совершена попытка присвоить этот номер,
        если это невозможно, то будет поднята ошибка.
        В случае, если параметр не передан, то номер будет сгенерирован автоматически

        :param number: Номер для ручной установки
        :return: Отформатированный номер аккаунта в строковом представлении

        :raise ValueError: Введённый вручную номер недоступен или некорректен
        """
        if number is None:
            number = self._inc()

        if isinstance(number, str):
            number = self.__extract_number(number)

        if number in self.__occupied_ids or number >= self.__limit:
            err = "Введённый вручную номер аккаунта недоступен"
            raise ValueError(err)

        if number < 0:
            err = "Итоговый номер не должен быть отрицательным"
            raise ValueError(err)

        self.__occupied_ids.add(number)

        return self.pattern.format(number)


In [17]:
import re
from abc import ABC

from homework1.core.exc import NegativeDepositError
from homework1.operations.operations import Operation, mark_operation

from homework1.accounts.account_number_manager import AccountNumbersManager


class Account(ABC):
    account_type: str

    def __init__(
        self,
        account_holder: str,
        *,
        account_number_manager: AccountNumbersManager,
        balance: float = 0.0,
        account_number: int | str | None = None,
        allowed_operations: set[str] | None = None,
        enable_balance_override: bool = False,
    ):
        """Базовый класс для всех аккаунтов.

        :param account_holder: Имя Фамилия владельца аккаунта
        :param account_type: Тип аккаунта. Указывается в наследниках класса
        :param account_number_manager: Менеджер номеров аккаунтов
        :param balance: Изначальный баланс аккаунта
        :param account_number: Номер аккаунта
        :param allowed_operations: Список доступных операций аккаунта. Указывается в наследниках класса
        """
        # Список доступных операций над данным аккаунтом
        self.__allowed_operations = {"deposit", "withdraw"} | (
            allowed_operations or set()
        )

        self.__holder = self.__validate_holder(account_holder)

        self.__is_balance_override_enabled = enable_balance_override
        self._balance = balance

        self.__account_number: str = account_number_manager.addgen(account_number)
        self._operations_history: list[Operation] = []

    def __validate_holder(self, value: str) -> str:
        """Валидатор Имени Фамилии владельца аккаунта

        :param value: Кандидат на значение
        :return: Отвалидированное значение в случае успешной валидации

        :raise ValueError: В случае провальной валидации
        """
        _title_word_patter = "[А-ЯA-Z][а-яa-z]+"
        _pattern = f"^{_title_word_patter}\\s{_title_word_patter}$"
        _pattern = re.compile(_pattern)

        if not isinstance(value, str) or not value:
            err = "Значение должно быть строкой"
            raise ValueError(err)

        if not re.fullmatch(_pattern, value):
            err = "Имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей"
            raise ValueError(err)

        return value

    def _validate_float(self, value: float) -> float:
        """Валидатор значения типа float

        :param value: Кандидат на значение
        :return: Отвалидированное значение в случае успешной валидации

        :raises ValueError: В случае провально валидации
        """
        if value is None or not isinstance(value, float):
            err = "Переданное значение должно быть валидным числом"
            raise ValueError(err)

        return value

    def _validate_deposit(self, value: float) -> float:
        """Валидатор значения для использовании в функции депозита

        :param value: Кандидат на значение
        :return: Отвалидированное значение в случае успешной валидации

        :raises ValueError: В случае провально валидации
        """
        value = self._validate_float(value)

        if value <= 0:
            err = "Сумма депозита должна быть больше 0"
            raise NegativeDepositError(err)

        return value

    def _validate_withdraw(self, value: float) -> float:
        """Валидатор значения для использования в функции снятия

        :param value: Кандидат на значение
        :return: Отвалидированное значение в случае успешной валидации

        :raises ValueError: В случае провально валидации
        """
        value = self._validate_float(value)

        if value < 0:
            err = "Сумма снятия должна быть больше 0"
            raise ValueError(err)

        if self._balance - value < 0:
            err = "Остаток на балансе после снятия должен быть больше или равен 0"
            raise ValueError(err)

        return value

    @property
    def allowed_operations(self) -> set[str]:
        """Список доступных операций над аккаунтом"""
        return self.__allowed_operations

    @property
    def account_number(self) -> str:
        """Номер аккаунта"""
        return self.__account_number

    @property
    def holder(self) -> str:
        """Имя Фамилия владельца аккаунта"""
        return self.__holder

    def get_balance(self) -> float:
        """Метод получения текущего баланса.
        Выполнен таким образом в соответствие с заданием
        """
        return self._balance

    def override_balance(self, balance: float) -> None:
        """Спец. метод для перезаписи баланса аккаунта без операции.
        Срабатывает только если установлен специальный флаг при создании аккаунта
        """
        if self.__is_balance_override_enabled:
            self._balance = self._validate_float(balance)

    def disable_balance_override(self) -> None:
        """Спец. метод для отключения возможности перезаписи баланса аккаунта без операции"""
        self.__is_balance_override_enabled = False

    def get_history(self) -> list[Operation]:
        """Метод получения истории операций.
        Выполнен таким образом в соответствие с заданием
        """
        return self._operations_history

    def add_operation_to_history(self, op: Operation) -> None:
        """Метод для добавления операции в историю из вне"""
        self._operations_history.append(op)

    def search_history(self, limit: int = 5) -> list[Operation]:
        """Поиск последних операций с сильнее всего повлиявших на баланс аккаунта.

        Под "влиянием" подразумевается наибольшая абсолютная разница
        между состояниями баланса перед применением операции и после неё.
        Учитываются только успешные операции

        :param limit: Лимит возвращаемых операций
        :return: Список из проделанных операций
        """
        sorted_by_date_and_balance = sorted(
            [op for op in self._operations_history if op.status == "success"],
            key=lambda _op: (
                abs(_op.balance_after - _op.balance_before),
                _op.created_at,
            ),
            reverse=True,
        )

        return sorted_by_date_and_balance[:limit]

    @mark_operation("deposit")
    def deposit(self, amount: float) -> None:
        """Операция внесения суммы на счёт

        :param amount: Сумма внесения

        :raise NegativeDepositError: Критическая ошибка, возникшая при попытке внесения негативной суммы
        """
        self._balance += self._validate_deposit(amount)

    @mark_operation("withdraw")
    def withdraw(self, amount: float) -> None:
        """Операция снятия суммы со счёта

        :param amount: Сумма снятия

        :raise ValueError: Возникает при ошибках валидации суммы или итогового баланса
        """
        self._balance -= self._validate_withdraw(amount)


class CheckingAccount(Account):
    account_type: str = "checking"

    def __init__(self, *args, **kwargs) -> None:
        """Аккаунт - Расчётный счёт"""
        # Не даём расширить список доступных операций
        kwargs.pop("allowed_operations", None)

        super().__init__(*args, **kwargs)


class SavingsAccount(Account):
    account_type: str = "savings"

    def __init__(self, *args, max_allowed_rate: float = 7.0, **kwargs) -> None:
        """Аккаунт - Сберегательный счёт

        :param max_allowed_rate: Максимальная процентная ставка по счёту
        """
        self._max_allowed_rate = max_allowed_rate

        # Не даём расширить список доступных операций
        kwargs.pop("allowed_operations", None)

        super().__init__(*args, **kwargs, allowed_operations={"interest"})

    def _validate_withdraw(self, value: float) -> float:
        """Валидатор значения для использования в функции снятия

        :param value: Кандидат на значение
        :return: Отвалидированное значение в случае успешной валидации

        :raises ValueError: В случае провально валидации
        """
        value = super()._validate_withdraw(value)

        if value > self._balance * 0.5:
            err = "Нельзя снять больше 50% от текущего баланса"
            raise ValueError(err)

        return value

    def _validate_interest(self, value: float) -> float:
        """Валидатор значения для использования в функции применения процентной ставки

        :param value: Кандидат на значение
        :return: Отвалидированное значение в случае успешной валидации

        :raises ValueError: В случае провально валидации
        """
        value = self._validate_float(value)

        if value <= 0 or value > self.max_allowed_rate:
            err = f"Размер процентной ставки не должен быть меньше 0 или больше максимальной процентной ставки max={self._max_allowed_rate}"
            raise ValueError(err)

        return value

    @property
    def max_allowed_rate(self) -> float:
        """Максимальная процентная ставка"""
        return self._max_allowed_rate

    @mark_operation("interest")
    def apply_interest(self, rate: float) -> None:
        """Операция применения процентной ставки на сумму счёта

        :param rate: Размер процентной ставки

        :raise ValueError: Возникает при ошибках валидации ставки
        """
        rate = self._validate_interest(rate)
        self._balance *= 1.0 + rate / 100.0


In [18]:
from typing import Generic, TypeVar

from homework1.accounts.accounts import Account, CheckingAccount, SavingsAccount

T = TypeVar("T", bound=Account)


class _AccountFormatter(Generic[T]):
    """Базовый форматтер для аккаунтов"""

    @classmethod
    def format(cls, account: T) -> str:
        """Метод форматирования аккаунта"""

        return f"""Номер аккаунта: {account.account_number}
Тип аккаунта: {account.account_type}
Владелец(ица) аккаунта: {account.holder}
Баланс аккаунта: {account.get_balance():03}
"""


class CheckingAccountFormatter(_AccountFormatter[CheckingAccount]):
    """Форматтер для аккаунтов типа - checking"""


class SavingsAccountFormatter(_AccountFormatter[SavingsAccount]):
    """Форматтер для аккаунтов типа - savings"""

    @classmethod
    def format(cls, account: SavingsAccount) -> str:
        return (
            super().format(account=account)
            + f"Максимальная ставка - {account.max_allowed_rate}%"
        )


class UniversalAccountFormatter:
    """Универсальный форматтер,
    который сам определяет тип аккаунта и применяет необходимый форматтер
    """

    # Таблица поддерживаемых типов аккаунтов
    _MAP = {
        CheckingAccount: CheckingAccountFormatter,
        SavingsAccount: SavingsAccountFormatter,
    }

    @classmethod
    def format(cls, account: Account) -> str:
        """Метод форматирования аккаунта"""

        for type_, formatter in cls._MAP.items():
            if isinstance(account, type_):
                return formatter.format(account=account)

        if isinstance(account, Account):
            return _AccountFormatter.format(account=account)

        raise NotImplementedError


In [19]:
"""Мне этот модуль очень не нравится, но в целом свою работу он выполняет"""

from homework1.accounts.account_number_manager import AccountNumbersManager
from homework1.accounts.accounts import Account, CheckingAccount, SavingsAccount
from homework1.operations.operations import Operation


class AccountRestoreError(Exception):
    """Произошла ошибка при восстановлении аккаунта"""


class AccountRestoreManager:
    """Класс, содержащий методы для восстановления аккаунтов из истории"""

    # Список поддерживающих восстановление типов аккаунтов
    _RESTORABLE_ACCOUNTS: tuple[type[Account], ...] = (CheckingAccount, SavingsAccount)

    @classmethod
    def restore(
        cls,
        account_type: str,
        account_number_manager: AccountNumbersManager,
        operations: list[Operation],
        account_number: int | str | None = None,
        strict: bool = True,
    ) -> CheckingAccount | SavingsAccount:
        """Метод восстановления аккаунта из истории

        :param account_type: Тип аккаунта
        :param account_number_manager: Менеджер номеров аккаунтов
        :param operations: История операций
        :param account_number: Номер аккаунта
        :param strict: Установка режима валидации.
        При установке в True - любая фатальная ошибка валидации будет приводить к невозможности создания аккаунта
        :return: Восстановленный аккаунт

        :raise AccountRestoreError: Если установлен параметр strict и произошла ошибка на уровне операций
        или если произошла ошибка на этапе создания аккаунта (strict=False не поможет в этом случае)
        """
        account_cls: type[Account] | None = None
        for acc_cls in cls._RESTORABLE_ACCOUNTS:
            if acc_cls.account_type == account_type:
                account_cls = acc_cls
                break

        if account_cls is None:
            err = "Аккаунт этого типа невозможно восстановить"
            raise ValueError(err)

        try:
            account = account_cls(
                account_holder="Unknown Account",
                account_number=account_number,
                account_number_manager=account_number_manager,
                enable_balance_override=True,
            )
            operations = cls.clean_history(account, operations)

            # Переопределяем стартовый баланс на основе самой первой операции из истории,
            # чтобы валидация шла на уровне согласованности данных
            account.add_operation_to_history(operations[0])
            account.override_balance(operations[0].balance_after)
            account.disable_balance_override()
        except Exception as e:
            err = f"Произошла ошибка при восстановлении аккаунта: {e}"
            raise AccountRestoreError(err) from e

        for op in operations[1:]:
            fn = getattr(account, op.name, None)

            # Очень костыльный метод, тк я бы завёл список вида <операция:метод> в каждом классе,
            # но это нужно много переписать, тк тогда я бы хотел сделать и автоматическое применение
            # декоратора для установки истории операции, а тогда почему бы и не переписать саму историю
            # на внешний класс-хранилище, но это долго...
            # В общем да, мне самому не нравится
            if not fn and account_cls == SavingsAccount and op.name == "interest":
                fn = account.apply_interest

            try:
                fn(op.value)

                balance = account.get_balance()

                if op.balance_after != balance:
                    err = f"Итоговый баланс после применения операции не соответствует балансу, указанному в истории ({balance} != {op.balance_after})"
                    raise ValueError(err)
            except ValueError as e:
                print(str(e))
            except Exception as e:
                if strict:
                    err = f"Произошла ошибка при восстановлении аккаунта: {e}"
                    raise AccountRestoreError(err) from e

                err = f"Произошла фатальная ошибка, из-за которой аккаунт не должен быть создан, но параметр {strict=}: {e}"
                print(err)

        return account

    @classmethod
    def clean_history(
        cls, account: Account, history: list[Operation]
    ) -> list[Operation]:
        """Крайне топорный метод очистки истории от априори невозможных типов операции над аккаунтом"""
        result = []

        for operation in history:
            if operation.name not in account.allowed_operations:
                continue

            result.append(operation)

        return result


# Далее пойдут примеры рыботы кода из папки `examples`

In [20]:
"""Модуль с примерами создания аккаунтов"""

from homework1.accounts.account_number_manager import AccountNumbersManager
from homework1.accounts.accounts import CheckingAccount, SavingsAccount
from homework1.accounts.formatters import UniversalAccountFormatter

if __name__ == "__main__":
    ANM = AccountNumbersManager()

    accounts = []

    accounts.append(CheckingAccount("Barbara Hoppkins", account_number_manager=ANM))
    accounts.append(CheckingAccount("Мария Рожкова", account_number_manager=ANM))

    try:
        accounts.append(CheckingAccount("ПростоМария", account_number_manager=ANM))
    except Exception as e:
        err = f"Создание аккаунта завершилось с ошибкой: {e}"
        print(err)

    accounts.append(CheckingAccount("Ирина Авдеева", account_number_manager=ANM))

    accounts.append(SavingsAccount("Алина Бережливая", account_number_manager=ANM))

    for acc in accounts:
        print(UniversalAccountFormatter.format(acc))


Создание аккаунта завершилось с ошибкой: Имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей
Номер аккаунта: ACC-0000
Тип аккаунта: checking
Владелец(ица) аккаунта: Barbara Hoppkins
Баланс аккаунта: 0.0

Номер аккаунта: ACC-0001
Тип аккаунта: checking
Владелец(ица) аккаунта: Мария Рожкова
Баланс аккаунта: 0.0

Номер аккаунта: ACC-0002
Тип аккаунта: checking
Владелец(ица) аккаунта: Ирина Авдеева
Баланс аккаунта: 0.0

Номер аккаунта: ACC-0003
Тип аккаунта: savings
Владелец(ица) аккаунта: Алина Бережливая
Баланс аккаунта: 0.0
Максимальная ставка - 7.0%


In [21]:
"""Модуль с примерами создания и отображения операций"""

from homework1.accounts.account_number_manager import AccountNumbersManager
from homework1.accounts.accounts import Account, CheckingAccount, SavingsAccount
from homework1.accounts.formatters import UniversalAccountFormatter
from homework1.operations.formatters import OperationsTableFormatter


def _perform_default_ops(account: Account) -> None:
    account.deposit(100.0)
    try:
        account.deposit(-100.0)
    except Exception as e:
        err = f"Операция завершилась с ошибкой: {e}"
        print(err)

    account.deposit(20.0)
    account.withdraw(15.0)

    try:
        account.withdraw(1000.0)
    except Exception as e:
        err = f"Операция завершилась с ошибкой: {e}"
        print(err)


def _make_checking_account(anm: AccountNumbersManager) -> CheckingAccount:
    account = CheckingAccount("Barbara Hoppkins", account_number_manager=anm)

    _perform_default_ops(account)

    return account


def _make_savings_account(anm: AccountNumbersManager) -> SavingsAccount:
    account = SavingsAccount("Алина Бережливая", account_number_manager=anm)

    _perform_default_ops(account)

    try:
        account.apply_interest(10.0)
    except Exception as e:
        err = f"Операция завершилась с ошибкой: {e}"
        print(err)

    account.apply_interest(5.0)

    return account


if __name__ == "__main__":
    ANM = AccountNumbersManager()
    operations_formatter = OperationsTableFormatter()

    for make_account_fn in (_make_checking_account, _make_savings_account):
        account = make_account_fn(ANM)

        print(UniversalAccountFormatter.format(account))
        print(operations_formatter.format(account.get_history()))


Операция завершилась с ошибкой: Сумма депозита должна быть больше 0
Номер аккаунта: ACC-0000
Тип аккаунта: checking
Владелец(ица) аккаунта: Barbara Hoppkins
Баланс аккаунта: 105.0

|----------+---------+--------+----------------+---------------+--------------------------+
|   name   | status  | value  | balance_before | balance_after |        created_at        |
|----------+---------+--------+----------------+---------------+--------------------------+
|deposit   |success  |   100.0|             0.0|          100.0|       2026-10-03 08:28:33|
|----------+---------+--------+----------------+---------------+--------------------------+
|deposit   |fail     |  -100.0|           100.0|          100.0|       2026-10-03 08:28:33|
|----------+---------+--------+----------------+---------------+--------------------------+
|deposit   |success  |    20.0|           100.0|          120.0|       2026-10-03 08:28:33|
|----------+---------+--------+----------------+---------------+-------------------

In [23]:
"""Модуль с примерами загрузки и выгрузки истории операций"""

from pathlib import Path

from homework1.operations.formatters import OperationsTableFormatter
from homework1.operations.serialization import (
    OperationsHistoryCSVFSStorageHandler,
    OperationsHistoryJSONFSStorageHandler,
)

SRC_DIRPATH = Path().resolve()
DATA_DIRPATH = SRC_DIRPATH / "data"
TRANSACTIONS_BASE_FILENAME = "transactions_dirty"

OUTPUT_DATA_DIRPATH = SRC_DIRPATH / "dist"

if __name__ == "__main__":
    if not OUTPUT_DATA_DIRPATH.exists():
        OUTPUT_DATA_DIRPATH.mkdir(exist_ok=True)

    _MAP = {
        ".csv": OperationsHistoryCSVFSStorageHandler,
        ".json": OperationsHistoryJSONFSStorageHandler,
    }

    _DUMPED_FILES_COUNTER = 0

    operations_formatter = OperationsTableFormatter()

    for suffix, handler_cls in _MAP.items():
        load_filepath = (DATA_DIRPATH / TRANSACTIONS_BASE_FILENAME).with_suffix(suffix)
        handler = handler_cls()

        history = handler.load(load_filepath)

        print(f"Файл по пути filepath={load_filepath} загружен")
        print(
            f"Вывод операций для типа: {suffix}. Эти операции прошли лишь базовую валидацию"
        )
        for *_, operations in history:
            print(operations_formatter.format(operations))

        dump_filepath = OUTPUT_DATA_DIRPATH / f"dump_{_DUMPED_FILES_COUNTER}{suffix}"
        handler.dump(dump_filepath, history)
        _DUMPED_FILES_COUNTER += 1

        print(f"Файл сохранён по пути filepath={dump_filepath}")

    print(f"Файлы сохранены в папке dirpath={OUTPUT_DATA_DIRPATH}. Файлы в папке:")
    print(*list(OUTPUT_DATA_DIRPATH.iterdir()), sep="\n")


Операцию не удалось распознать:
{'account_number': 'ACC-100001',
 'account_type': 'checking',
 'amount': '129.0',
 'balance_after': '3345.0',
 'date': '28/09/2025 22:17',
 'operation': 'deposit',
 'status': 'success'}
Операцию не удалось распознать:
{'account_number': 'ACC-100001',
 'account_type': 'checking',
 'amount': '',
 'balance_after': '4039.0',
 'date': '2025-09-29 22:17:26',
 'operation': 'withdraw',
 'status': 'success'}
Операцию не удалось распознать:
{'account_number': 'ACC-100001',
 'account_type': 'checking',
 'amount': '793.0',
 'balance_after': '5350.0',
 'date': '2025-17-34 12:00:00',
 'operation': 'deposit',
 'status': 'success'}
Операцию не удалось распознать:
{'account_number': 'ACC-100001',
 'account_type': 'checking',
 'amount': '198.0',
 'balance_after': '5152.0',
 'date': '06/10/2025 22:17',
 'operation': 'withdraw',
 'status': 'success'}
Операцию не удалось распознать:
{'account_number': 'ACC-100001',
 'account_type': 'checking',
 'amount': '111.0',
 'balance_a

In [ ]:
"""Модуль с примерами восстановления аккаунтов из истории"""

from pathlib import Path

from homework1.accounts.account_number_manager import AccountNumbersManager
from homework1.accounts.formatters import UniversalAccountFormatter
from homework1.accounts.restoration import AccountRestoreManager
from homework1.operations.formatters import OperationsTableFormatter
from homework1.operations.serialization import (
    AccountOperations,
    OperationsHistoryCSVFSStorageHandler,
)

SRC_DIRPATH = Path().resolve()
DATA_DIRPATH = SRC_DIRPATH / "data"
TRANSACTIONS_FILEPATH = DATA_DIRPATH / "transactions_dirty.csv"


def _restore_accounts(history: list[AccountOperations], strict: bool = True) -> None:
    anm = AccountNumbersManager()

    accounts = []
    for account_number, account_type, operations in history:
        try:
            account = AccountRestoreManager.restore(
                account_type=account_type,
                account_number=account_number,
                operations=operations,
                account_number_manager=anm,
                strict=strict,
            )

            accounts.append(account)
        except Exception as e:
            err = f"Невозможно восстановить аккаунт {account_number=} из-за фатальной ошибки {e.__class__.__name__}: {e}"
            print(err)

    print("Восстановленные аккаунты:")
    for account in accounts:
        print(UniversalAccountFormatter.format(account))
        print(operations_formatter.format(account.get_history()))


if __name__ == "__main__":
    operations_formatter = OperationsTableFormatter()

    csv_history_handler = OperationsHistoryCSVFSStorageHandler()
    history = csv_history_handler.load(TRANSACTIONS_FILEPATH)

    print(
        "Восстановление аккаунтов в строгом режиме. Любая фатальная ошибка приведёт к невозможности восстановления аккаунта"
    )
    _restore_accounts(history, strict=True)

    print(
        "Восстановление аккаунтов в упрощённом режиме. Любые фатальные ошибки пропускаются"
    )
    _restore_accounts(history, strict=False)
